# Semana 4 — Taller práctico: funciones, modularización y GitHub

**Asignatura:** Análisis de Datos con Python  
**Tipo de evidencia:** Notebook con ejercicios resueltos + repositorio inicial en GitHub  

## Propósito del taller

Aplicar funciones, modularización básica, comprensión de listas y diccionarios, manejo de errores y buenas prácticas de programación en un caso sencillo de análisis de datos.

## Entregables

1. Notebook completo con los ejercicios resueltos.
2. Repositorio inicial en GitHub con estructura organizada.
3. Archivo `README.md` con descripción del repositorio.
4. Evidencia del enlace al repositorio en la última sección del notebook.

## Instrucciones

Complete las celdas marcadas con `TODO`.

In [14]:
import pandas as pd

# 1. Cargar el dataset desde el archivo CSV del proyecto
df_raw = pd.read_csv('mecatronics_dataset.csv')

# 2. Convertir las filas del CSV al formato de la lista `registros_crudos`
registros_crudos = []
for _, row in df_raw.iterrows():
    registros_crudos.append({
        "id": f"M-{int(row['time']):03d}",
        "voltaje": "12.0",  # Voltaje nominal de prueba para calcular la potencia
        "corriente": str(round(row["current_A"], 2)),
        "temperatura": str(round(row["temperature_C"], 2))
    })

print(f"Se cargaron {len(registros_crudos)} registros desde mecatronics_dataset.csv.")


Se cargaron 1000 registros desde mecatronics_dataset.csv.


## Parte 1 — Función para convertir valores

Cree una función llamada `convertir_a_float(valor)` que intente convertir un valor a número decimal.

La función debe retornar:

- El número convertido, si la conversión es posible.
- `None`, si el valor no se puede convertir.

In [15]:
def convertir_a_float(valor):
    if valor is None:
        return None
    try:
        return float(valor)
    except (ValueError, TypeError):
        return None

# Pruebas sugeridas
print(convertir_a_float("12.5"))  # Output: 12.5
print(convertir_a_float("error")) # Output: None
print(convertir_a_float(""))      # Output: None
print(convertir_a_float(None))    # Output: None

12.5
None
None
None


## Parte 2 — Función para calcular potencia

Cree una función llamada `calcular_potencia(voltaje, corriente)`.

La función debe:

- Retornar `None` si voltaje o corriente son `None`.
- Retornar `None` si voltaje o corriente son negativos.
- Retornar `voltaje * corriente` en los demás casos.

In [2]:
def calcular_potencia(voltaje, corriente):
    if voltaje is None or corriente is None:
        return None
    if voltaje < 0 or corriente < 0:
        return None
    return voltaje * corriente

# Pruebas sugeridas
print(calcular_potencia(12, 2))   # Output: 24
print(calcular_potencia(None, 2)) # Output: None
print(calcular_potencia(12, -1))  # Output: None

24
None
None


## Parte 3 — Función para clasificar temperatura

Cree una función llamada `clasificar_temperatura(temp)` con los siguientes criterios:

- Si `temp` es `None`: `"Dato inválido"`
- Si `temp < 40`: `"Normal"`
- Si `40 <= temp < 50`: `"Precaución"`
- Si `temp >= 50`: `"Alerta"`

In [3]:
def clasificar_temperatura(temp):
    if temp is None:
        return "Dato inválido"
    if temp < 40:
        return "Normal"
    elif 40 <= temp < 50:
        return "Precaución"
    else:
        return "Alerta"

# Pruebas sugeridas
print(clasificar_temperatura(35))   # Output: Normal
print(clasificar_temperatura(45))   # Output: Precaución
print(clasificar_temperatura(52))   # Output: Alerta
print(clasificar_temperatura(None)) # Output: Dato inválido

Normal
Precaución
Alerta
Dato inválido


## Parte 4 — Limpieza y procesamiento de registros

Procese la lista `registros_crudos` y cree una nueva lista llamada `registros_limpios`.

Cada registro limpio debe contener:

- `id`
- `voltaje`
- `corriente`
- `temperatura`
- `potencia`
- `estado_temperatura`

Use las funciones creadas.

In [21]:
registros_limpios = []

for registro in registros_crudos:
    v = convertir_a_float(registro.get("voltaje"))
    i = convertir_a_float(registro.get("corriente"))
    t = convertir_a_float(registro.get("temperatura"))

    p = calcular_potencia(v, i)
    estado = clasificar_temperatura(t)

    registros_limpios.append({
        "id": registro.get("id"),
        "voltaje": v,
        "corriente": i,
        "temperatura": t,
        "potencia": p,
        "estado_temperatura": estado
    })

print(f"Registros limpios procesados: {len(registros_limpios)}")

Registros limpios procesados: 1000


## Parte 5 — Comprensión de listas

Use comprensión de listas para obtener:

1. Lista de potencias válidas.
2. Lista de temperaturas válidas.
3. Lista de motores en alerta.

In [22]:
potencias_validas = [r["potencia"] for r in registros_limpios if r["potencia"] is not None]
temperaturas_validas = [r["temperatura"] for r in registros_limpios if r["temperatura"] is not None]
motores_alerta = [r["id"] for r in registros_limpios if r["estado_temperatura"] == "Alerta"]

print("Potencias válidas procesadas:", len(potencias_validas))
print("Temperaturas válidas procesadas:", len(temperaturas_validas))
print("Motores en alerta:", motores_alerta)

Potencias válidas procesadas: 1000
Temperaturas válidas procesadas: 1000
Motores en alerta: []


## Parte 6 — Comprensión de diccionarios

Cree un diccionario llamado `estado_por_motor` donde:

- La clave sea el `id` del motor.
- El valor sea el `estado_temperatura`.

In [23]:
estado_por_motor = {r["id"]: r["estado_temperatura"] for r in registros_limpios}

## Parte 7 — Resumen automático

Cree una función llamada `generar_resumen(registros)` que retorne un diccionario con:

- `total_registros`
- `registros_validos_potencia`
- `potencia_promedio`
- `temperatura_promedio`
- `cantidad_alertas`
- `cantidad_precauciones`
- `cantidad_normales`

In [24]:
def generar_resumen(registros):
    potencias = [r["potencia"] for r in registros if r["potencia"] is not None]
    temperaturas = [r["temperatura"] for r in registros if r["temperatura"] is not None]

    total_registros = len(registros)
    registros_validos_potencia = len(potencias)

    potencia_promedio = sum(potencias) / len(potencias) if potencias else 0.0
    temperatura_promedio = sum(temperaturas) / len(temperaturas) if temperaturas else 0.0

    cantidad_alertas = sum(1 for r in registros if r["estado_temperatura"] == "Alerta")
    cantidad_precauciones = sum(1 for r in registros if r["estado_temperatura"] == "Precaución")
    cantidad_normales = sum(1 for r in registros if r["estado_temperatura"] == "Normal")

    return {
        "total_registros": total_registros,
        "registros_validos_potencia": registros_validos_potencia,
        "potencia_promedio": round(potencia_promedio, 2),
        "temperatura_promedio": round(temperatura_promedio, 2),
        "cantidad_alertas": cantidad_alertas,
        "cantidad_precauciones": cantidad_precauciones,
        "cantidad_normales": cantidad_normales
    }

resumen = generar_resumen(registros_limpios)
resumen

{'total_registros': 1000,
 'registros_validos_potencia': 1000,
 'potencia_promedio': 24.15,
 'temperatura_promedio': 20.16,
 'cantidad_alertas': 0,
 'cantidad_precauciones': 0,
 'cantidad_normales': 1000}

## Parte 8 — Visualización como tabla

Convierta `registros_limpios` en un `DataFrame` de Pandas.

In [25]:
import pandas as pd

df = pd.DataFrame(registros_limpios)
df

,id,voltaje,corriente,temperatura,potencia,estado_temperatura
0,M-000,12.0,1.93,20.40,23.16,Normal
1,M-001,12.0,2.00,19.99,24.00,Normal
2,M-002,12.0,1.94,20.72,23.28,Normal
3,M-003,12.0,2.00,21.52,24.00,Normal
4,M-004,12.0,1.85,20.21,22.20,Normal
...,...,...,...,...,...,...
995,M-995,12.0,2.30,24.11,27.60,Normal
996,M-996,12.0,2.32,25.83,27.84,Normal
997,M-997,12.0,2.17,24.95,26.04,Normal
998,M-998,12.0,2.32,24.02,27.84,Normal


## Parte 9 — Buenas prácticas

Revise su notebook y verifique:

- Los nombres de variables son claros.
- Las funciones tienen una tarea específica.
- El código no está repetido innecesariamente.
- Las celdas están ejecutadas en orden.
- Hay explicaciones en texto.

## Parte 10 — Repositorio inicial en GitHub

Cree un repositorio en GitHub con el nombre:

**analisis-datos-python-apellido-nombre**

Estructura mínima esperada:

```text
analisis-datos-python-apellido-nombre/
│
├── README.md
├── requirements.txt
├── notebooks/
│   ├── semana_02_introduccion.ipynb
│   ├── semana_03_fundamentos_python.ipynb
│   └── semana_04_funciones_modularizacion.ipynb
│
├── data/
│   ├── raw/
│   └── processed/
│
├── src/
│   └── funciones.py
│
└── reports/
    └── figuras/
```

Enlace al repositorio:

Pegue aquí el enlace de su repositorio.

**Enlace al repositorio:**  
https://github.com/ChristianDiazB/analisis-datos-python-christian-diaz

## Parte 11 — Reflexión final

Responda:

1. ¿Qué ventaja tuvo usar funciones en el procesamiento de los registros?
2. ¿Por qué es importante manejar errores antes de analizar datos?
3. ¿Qué función del taller considera más útil?
4. ¿Qué elemento del repositorio ayuda más a la reproducibilidad?

**Respuesta:**  
1. Permiten encapsular la lógica de validación y cálculo, evitando duplicar código al procesar múltiples registros e independizando la transformación de los datos de la estructura del bucle principal.
2. Evita fallos en tiempo de ejecución y garantiza que las métricas finalesse calculen sobre información limpia y consistente.
3. convertir_a_float, ya que actúa como la primera línea de defensa sanitaria ante datos inconsistentes, texto mal formateado o valores faltantes.
4. El archivo requirements.txt y la estructura organizada de carpetas, garantizan que cualquier otro desarrollador o investigador pueda recrear el mismo entorno execution e independizar las funciones puras.